# Grid Search Demo (All 3 Models)

This notebook runs hyperparameter grid search for ItemKNN, SVD, and LightFM using the shared train/validation split.

In [13]:
from pathlib import Path

import pandas as pd

from src.evaluation.grid_search import GridSearchConfig
from src.evaluation.grid_search import RecommenderGridSearch

In [14]:
# Resolve project root from notebook folder.
project_root_path = Path.cwd()
if not (project_root_path / "src").exists():
    project_root_path = project_root_path.parent

train_ratings_path = project_root_path / "data" / "processed" / "notebook_demo" / "ratings_train_split.csv"
validation_ratings_path = project_root_path / "data" / "processed" / "notebook_demo" / "ratings_validation_split.csv"
movies_features_path = project_root_path / "data" / "processed" / "movies_cleaned.csv"
grid_output_directory_path = project_root_path / "data" / "processed" / "grid_search" / "notebook_run"

train_dataframe = pd.read_csv(train_ratings_path)
validation_dataframe = pd.read_csv(validation_ratings_path)
movies_dataframe = pd.read_csv(movies_features_path)

print(f"Train rows: {len(train_dataframe)}")
print(f"Validation rows: {len(validation_dataframe)}")
print(f"Movies rows: {len(movies_dataframe)}")

Train rows: 88017
Validation rows: 9784
Movies rows: 9742


In [15]:
# Configure search for all three models.
# Set maximum_trials_per_model=None for full exhaustive runs.
search_config = GridSearchConfig(
    selected_model_names=[
        "itemknn",
        # "svd",
        # "lightfm"
    ],
    metric_name="ndcg_at_k",
    number_of_recommendations=10,
    relevance_threshold=4.0,
    maximum_trials_per_model=3, # when testing this, i would recommend turning this on since otherwise you could end up with long runtimes.
    output_directory_path=grid_output_directory_path,
)
search_runner = RecommenderGridSearch(search_config=search_config)

In [16]:
# Run grid search and collect best trial per model.
model_results = search_runner.run(
    train_dataframe=train_dataframe,
    validation_dataframe=validation_dataframe,
    movies_dataframe=movies_dataframe,
)

print(f"Saved artifacts in: {grid_output_directory_path}")
for model_result in model_results:
    metric_value = getattr(model_result.best_trial.evaluation_result, model_result.metric_name)
    print(
        f"Model={model_result.model_name}, "
        f"best_trial={model_result.best_trial.trial_index}, "
        f"{model_result.metric_name}={metric_value:.4f}, "
        f"params={model_result.best_trial.parameter_values}"
    )

In [17]:
# Load saved best-result files for quick review.
best_rows = []
for model_name in ["itemknn", "svd", "lightfm"]:
    best_json_path = grid_output_directory_path / model_name / "best_result.json"
    if not best_json_path.exists():
        continue
    best_payload = pd.read_json(best_json_path, typ="series")
    best_rows.append(best_payload)


if best_rows:
    results = pd.DataFrame(best_rows)
    # Extract NDCG into a top-level column for easier sorting and filtering.
    results["ndcg_at_k"] = results["best_metrics"].apply(
        lambda metrics_payload: metrics_payload.get("ndcg_at_k") if isinstance(metrics_payload, dict) else None
    )
else:
    results = None
    print("No best_result.json files found yet.")

results

,model_name,metric_name,best_trial_index,best_parameters,best_metrics,ndcg_at_k
0,itemknn,ndcg_at_k,2,"{'number_of_neighbors': 20, 'minimum_neighbors...","{'rmse_value': 0.9153824198828011, 'mae_value'...",0.000335
1,svd,ndcg_at_k,1,"{'number_of_factors': 20, 'number_of_epochs': ...","{'rmse_value': 0.9390070093789241, 'mae_value'...",0.180274
2,lightfm,ndcg_at_k,3,"{'number_of_components': 16, 'number_of_epochs...","{'rmse_value': 6.006674191613492, 'mae_value':...",0.017261


In [18]:
# Show all-trials table for one model as an example.
svd_trials_path = grid_output_directory_path / "svd" / "all_trials.csv"
pd.read_csv(svd_trials_path).head()

,model_name,trial_index,param_number_of_factors,param_number_of_epochs,param_learning_rate_all,param_regularization_all,param_random_seed,rmse_value,mae_value,precision_at_k,recall_at_k,ndcg_at_k,novelty_at_k,diversity_at_k,item_coverage_at_k,intra_list_similarity_at_k,item_to_history_distance_at_k,serendipity_at_k
0,svd,1,20,10,0.002,0.02,42,0.939007,0.720472,0.157576,0.075267,0.180274,9.061753,0.694705,0.003599,0.305295,0.179746,0.179746
1,svd,2,20,10,0.002,0.05,42,0.938853,0.720718,0.154545,0.073582,0.177050,9.070637,0.693964,0.003281,0.306036,0.176645,0.176645
2,svd,3,20,10,0.002,0.10,42,0.938948,0.721370,0.152525,0.072780,0.174027,9.084985,0.691215,0.003281,0.308785,0.176125,0.176125
